# Lab Orpheus - Stage 1-2 manifest

Open-weight local: `orpheus-speech` + vLLM. Do **not** use paid API package `orpheus-tts`.

## Official deploy (upstream)
- README: https://github.com/canopyai/Orpheus-TTS
- Install: `pip install orpheus-speech` then **`pip install vllm==0.7.3`**
- Model: `canopylabs/orpheus-tts-0.1-finetune-prod`
- Voices: tara, leah, jess, leo, dan, mia, zac, zoe
- Official Colab: https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N

## Why install is slow
Downloading large wheels (vLLM/CUDA) and later multi-GB model weights. Pip conflict spam from Colab preinstalled packages is mostly noise.

## Errors seen in this project notebook outputs
1. libcudart.so.13 - vLLM CUDA 13 wheel on Colab CUDA 12
2. numpy.dtype size changed - mixed NumPy binaries (need Restart)
3. PIL `_Ink` import error - Pillow/torchvision skew after pip churn
4. cuda_available False - GPU runtime not attached
5. missing lab_render_report.json - only because render never started

**Always:** Runtime GPU on. After broken imports: Restart session (or Delete runtime).


In [ ]:
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


In [ ]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file(), "not in repo root"
print("CWD", os.getcwd())


In [ ]:
# INSTALL - run once per fresh runtime. Heavy packages stay on Colab only.
import sys
import subprocess

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("run:", " ".join(cmd))
    subprocess.check_call(cmd)

print("python", sys.version)

# light control plane
pip_install("-e", ".[dev]")

# remove paid API client if present
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "orpheus-tts"])

# official open-weight path
pip_install("orpheus-speech")
pip_install("vllm==0.7.3")

# stabilize Colab ABI / PIL breaks seen in lab logs
pip_install("--force-reinstall", "numpy==1.26.4", "pillow==10.4.0", "soundfile")

import numpy as np
import torch
print("numpy", np.__version__)
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: enable GPU runtime before Orpheus")

try:
    import vllm
    from orpheus_tts import OrpheusModel
    print("vllm", getattr(vllm, "__version__", "?"))
    print("OrpheusModel import: OK")
except Exception as e:
    print("IMPORT FAILED", type(e).__name__, e)
    print("NEXT: Runtime -> Restart session, re-run clone+install.")
    print("If still broken: Disconnect and delete runtime, new GPU, full reinstall.")
    print("Official Colab:", "https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N")
    raise


In [ ]:
# After Restart: quick import check (skip reinstall if packages already OK)
import torch
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
from orpheus_tts import OrpheusModel
import vllm
print("vllm", getattr(vllm, "__version__", "?"), "OrpheusModel OK")


In [ ]:
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("lab_audio/orpheus_part1").mkdir(parents=True, exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json
m = json.loads(Path("outputs/part1_manifest.json").read_text())
assert m["validation"]["valid"] is True
print("n_requests", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], u["spoken_text"][:90])


In [ ]:
import json
import os
from pathlib import Path

OUT = Path("lab_audio/orpheus_part1")
REPORT = OUT / "lab_render_report.json"
OUT.mkdir(parents=True, exist_ok=True)

# smoke 4 segments; remove --limit 4 for full dialogue
cmd = (
    "python scripts/lab_render_orpheus_from_manifest.py "
    "--manifest outputs/part1_manifest.json "
    "--voice-map configs/voice_maps/orpheus_en_part1.json "
    "--out-dir lab_audio/orpheus_part1 "
    "--temperature 0.6 --repetition-penalty 1.3 --limit 4"
)
print(cmd)
ret = os.system(cmd)
print("exit", ret)
if not REPORT.is_file():
    raise FileNotFoundError(
        "lab_render_report.json missing: render never finished. "
        "Fix Orpheus import first (install cell + Restart)."
    )
report = json.loads(REPORT.read_text())
print("ok", report["ok_count"], "fail", report["fail_count"])
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert REPORT_PATH.is_file(), "run render first"
report = json.loads(REPORT_PATH.read_text())
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent
print(report.get("transcript_id"), "n=", len(segments))
for i, seg in enumerate(segments, 1):
    fname = seg.get("output_filename")
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(f"[{i}/{len(segments)}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('backend_voice_id')}")
    print("DISPLAY:", seg.get("display_text", ""))
    print("SPOKEN :", seg.get("spoken_text", ""))
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print("missing", fname)


In [ ]:
from pathlib import Path
import json
report = json.loads(Path("lab_audio/orpheus_part1/lab_render_report.json").read_text())
reviews = {s["segment_id"]: {"content_match": "", "notes": ""} for s in report.get("segments") or []}
# reviews["seg_0001"]["content_match"] = "yes"
filled = []
for s in report.get("segments") or []:
    h = reviews.get(s["segment_id"]) or {}
    filled.append({
        "segment_id": s.get("segment_id"),
        "display_text": s.get("display_text"),
        "spoken_text": s.get("spoken_text"),
        "backend_voice_id": s.get("backend_voice_id"),
        "output_filename": s.get("output_filename"),
        "content_match": (h.get("content_match") or "").strip().lower(),
        "notes": h.get("notes", ""),
    })
Path("lab_audio/orpheus_part1/segment_review_filled.json").write_text(
    json.dumps(filled, ensure_ascii=False, indent=2) + "\n"
)
print("saved", len(filled))


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display
assert Path("lab_audio/orpheus_part1/lab_render_report.json").is_file()
!python scripts/lab_concat_segment_wavs.py --report lab_audio/orpheus_part1/lab_render_report.json --out lab_audio/orpheus_part1/part1_full.wav --gap-mode adaptive
full = Path("lab_audio/orpheus_part1/part1_full.wav")
print(json.loads(Path("lab_audio/orpheus_part1/part1_full.concat.json").read_text()).get("duration_sec"))
if full.is_file():
    display(Audio(filename=str(full)))


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

!python scripts/lab_survey_orpheus_voices.py --manifest outputs/part1_manifest.json --inventory configs/voice_maps/orpheus_voice_inventory.json --preset en_shortlist --out-dir lab_audio/orpheus_voice_survey --event-ids e004,e006,e008,e011

rp = Path("lab_audio/orpheus_voice_survey/voice_survey_report.json")
assert rp.is_file(), "survey failed - check import/GPU"
report = json.loads(rp.read_text())
print("ok/fail", report.get("ok_count"), report.get("fail_count"), report.get("voices"))
by = defaultdict(list)
for r in report.get("renders") or []:
    by[r.get("event_id")].append(r)
for eid, items in by.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    print("=" * 72)
    print("LINE", eid)
    if items:
        print("DISPLAY", items[0].get("display_text"))
        print("SPOKEN ", items[0].get("spoken_text"))
    for it in items:
        print("---", it.get("voice_id"), it.get("status"))
        p = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if p.is_file():
            display(Audio(filename=str(p)))


Checklist: GPU on? import OK? codes clear? vs Kokoro? no script edits? Lab verdict only - not_selected.
